# Calculating Lin's Concordance Correlation Coefficient for predicted volumes/thicknesses

In [ ]:
# imports
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import SimpleITK as sitk

## CCC function

In [ ]:
def ccc(y_true, y_pred):
    """
    Concordance Correlation Coefficient
    """
    # means
    m1 = np.mean(y_true)
    m2 = np.mean(y_pred)

    # variances
    v1 = np.var(y_true, ddof=1)
    v2 = np.var(y_pred, ddof=1)

    # covariance
    cov = np.cov(y_true, y_pred, ddof=1)[0][1]

    # concordance correlation coefficient
    ccc = 2 * cov / (v1 + v2 + (m1 - m2) ** 2)

    return ccc

## MAE Function

In [ ]:
def mean_absolute_error(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

In [ ]:
# Mean average percentage error
def mean_absolute_percentage_error(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [ ]:
# rmse coefficient of variation (scaled by mean to become dimensionless)
def rmse_cv(y_true, y_pred):
    return 100 * np.sqrt(np.mean((y_true - y_pred) ** 2)) / np.mean(y_true)

## Function to calculate average thickness of segmentation mask

In [ ]:
def average_thickness(mask: np.array, voxel_spacing: np.array):
    """
    Calculate the average thickness of a 3d mask (numpy array)
    """
    # total volume
    volume = np.sum(mask)

    # number of columns that are non-zero in x,z plane
    num_columns = np.count_nonzero(np.sum(mask, axis=(1)))

    # average thickness
    thickness = volume / num_columns

    # convert to mm
    thickness = thickness * voxel_spacing[1]

    return thickness


In [ ]:
# path to predicted data
data_path = '../nnUNet_data/nnUNet_raw/Dataset361_Menisci/iwoai_internal_results/'

# list subfolders at this path
subfolders = [f.name for f in os.scandir(data_path) if f.is_dir()]
print(subfolders)

In [ ]:
# what metrics are there?
json_path = os.path.join(data_path, 'clahe_fold_predsTs/fold0', 'summary.json')

with open(json_path) as f:
    data = json.load(f)

# extract all metrics
data["metric_per_case"][0]

In [ ]:
# functions to take json file and return array of segmentation volumes
def get_seg_vols(json_file):
    with open(json_file) as f:
        data = json.load(f)

    # Extract Dice scores from "metric_per_case"
    vols = [
        case["metrics"]["1"]["n_pred"] for case in data["metric_per_case"]
    ]

    return vols


def get_gt_vols(json_file):
    with open(json_file) as f:
        data = json.load(f)

    # Extract Hausdorff distances from "metric_per_case"
    vols = [
        case["metrics"]["1"]["n_ref"] for case in data["metric_per_case"]
    ]

    return vols

In [ ]:
# discount folders that start with ResEnc
subfolders = [f for f in subfolders if not f.startswith("ResEnc")]

# discount hist match
subfolders = [f for f in subfolders if "hist_match" not in f]

subfolders

In [ ]:
# reorder the subfolders using the order of the methods (starts with)
method_order = ["zscore", "rescale", "clip", "hist_eq", "clahe", "nyul", "gmm"]

# Sort by checking which method each subfolder starts with
subfolders = sorted(
    subfolders,
    key=lambda x: next((method_order.index(m) for m in method_order if x.lower().startswith(m.lower())), float('inf'))
)
subfolders

In [ ]:
# for each folder, get segmentation volumes and add to pandas dataframe

# create empty dataframe
internal_df = pd.DataFrame()

for i, folder in enumerate(subfolders):
    json_path = os.path.join(data_path, folder, 'summary.json')

    if i == 0:
        gt_vols = get_gt_vols(json_path)
        internal_df['GT'] = gt_vols

    seg_vols = get_seg_vols(json_path)
    internal_df[folder] = seg_vols

column_names = [
    "GT",
    "Zscore",
    "Rescale",
    "Clip Rescale",
    "Hist Eq",
    "Clahe",
    "Nyul",
    "GMM"
]

internal_df.columns = column_names

internal_df.describe()

In [ ]:
internal_df.head()

In [ ]:
# plot pairwise correlation matrix of dice scores, to 3 decimal places
plt.figure(figsize=(10, 6))
sns.heatmap(internal_df.corr('pearson'), annot=True, cmap='coolwarm', fmt=".3f")
plt.title("Pearson correlation matrix of Dice scores")
plt.show()

In [ ]:
# plot bland-altman for zscore vs rescale (means and differences)
import statsmodels.api as sm
sm.graphics.mean_diff_plot(internal_df["GT"], internal_df["GMM"])
#plt.savefig('bland_altman_zscore_rescale.eps', format='eps', bbox_inches='tight')
plt.show()

In [ ]:
# calculate CCC and MAE for each method
ccc_scores = []
mae_scores = []

for method in column_names[1:]:
    ccc_scores.append(ccc(internal_df["GT"], internal_df[method]))
    mae_scores.append(mean_absolute_error(internal_df["GT"], internal_df[method]))

# create dataframe of scores
scores_df = pd.DataFrame({
    "Method": column_names[1:],
    "CCC": ccc_scores,
    "MAE": mae_scores
})

scores_df

## Do average thickness now

In [ ]:
# load in first mask of first method to test
import glob

# get all mask paths
masks_paths = glob.glob(os.path.join(data_path, subfolders[0], 'fold0') + '/*.nii.gz')
masks_paths.sort()
masks_paths[:5]

In [ ]:
# load in first mask
mask = sitk.ReadImage(masks_paths[0])

# print mask info
print(mask.GetSize())
print(mask.GetSpacing())

# convert mask to numpy array
mask_np = sitk.GetArrayFromImage(mask)
print(mask_np.shape)

# get spacing in numpy array format and reverse to match sitk to numpy conversion
np_spacing = np.array(mask.GetSpacing())[::-1]
iwoai_spacing = np_spacing.copy()
print(np_spacing)

# show slice of mask
plt.imshow(mask_np[120,...])
plt.show()

# show mask collapsed in up-down direction
collapsed_mask = np.sum(mask_np, axis=1)
plt.imshow(collapsed_mask)
plt.show()

# number of columns that are non-zero in x,z plane
num_columns = np.count_nonzero(collapsed_mask)

# total volume
volume = np.sum(collapsed_mask)

# average thickness
thickness = volume / num_columns
thickness = thickness * np_spacing[1]
print("Average Thickness: {:.2f} mm".format(thickness))

In [ ]:
# check function
average_thickness(mask_np, np_spacing).round(2)

In [ ]:
# calculate average thickness for all GT masks
gt_masks_paths = glob.glob(os.path.join(data_path, '../labelsTs', '*.nii.gz'))
gt_masks_paths.sort()

average_thicknesses = []

for mask_path in gt_masks_paths:
    mask = sitk.ReadImage(mask_path)
    mask_np = sitk.GetArrayFromImage(mask)
    np_spacing = np.array(mask.GetSpacing())[::-1]
    thickness = average_thickness(mask_np, np_spacing)
    average_thicknesses.append(thickness)

# create dataframe of average thicknesses
thickness_df = pd.DataFrame({
    "GT": average_thicknesses
})

from tqdm import tqdm

# now calculate average thickness for all predicted masks of all methods
for folder in tqdm(subfolders):
    masks_paths = glob.glob(os.path.join(data_path, folder) + '/*.nii.gz')
    masks_paths.sort()

    average_thicknesses = []

    for mask_path in masks_paths:
        mask = sitk.ReadImage(mask_path)
        mask_np = sitk.GetArrayFromImage(mask)
        np_spacing = np.array(mask.GetSpacing())[::-1]
        thickness = average_thickness(mask_np, np_spacing)
        average_thicknesses.append(thickness)

    thickness_df[folder] = average_thicknesses

In [ ]:
# rename columns
thickness_df.columns = column_names

# describe dataframe
thickness_df.describe()

In [ ]:
# do CCC and MAE for average thickness
ccc_scores = []
mae_scores = []

for method in column_names[1:]:
    ccc_scores.append(ccc(thickness_df["GT"], thickness_df[method]))
    mae_scores.append(mean_absolute_error(thickness_df["GT"], thickness_df[method]))

# create dataframe of scores
thickness_scores_df = pd.DataFrame({
    "Method": column_names[1:],
    "CCC": ccc_scores,
    "MAE": mae_scores
})

thickness_scores_df

# Do External Validation on whole of SKMTEA

In [ ]:
data_path = '../nnUNet_data/nnUNet_raw/Dataset361_Menisci/skmtea_external_results/'

# get only the subfolders that end in _all_skmtea
subfolders = [f.name for f in os.scandir(data_path) if f.is_dir() and f.name.endswith("_all_skmtea")]

# discount images and labels folders (starts with)
subfolders = [f for f in subfolders if not f.startswith("images") and not f.startswith("labels") 
              and not f.startswith("ResEnc") and not f.startswith("clahe_hist_match")
              and not f.startswith("zscore_postproc") and "hist_match" not in f]
subfolders

In [ ]:
# reorder the subfolders using the order of the methods (starts with)
method_order = ["zscore", "rescale", "clip", "hist_eq", "clahe", "nyul", "gmm"]
# Sort by checking which method each subfolder starts with
subfolders = sorted(
    subfolders,
    key=lambda x: next((method_order.index(m) for m in method_order if x.lower().startswith(m.lower())), float('inf'))
)
subfolders

In [ ]:
# for each folder, get dice scores and add to pandas dataframe

# create empty dataframe
external_all_df = pd.DataFrame()

for i, folder in enumerate(subfolders):
    json_path = os.path.join(data_path, folder, 'summary.json')

    if i == 0:
        gt_vols = get_gt_vols(json_path)
        external_all_df['GT'] = gt_vols
    
    seg_vols = get_seg_vols(json_path)
    external_all_df[folder] = seg_vols

external_all_df.head()

In [ ]:
# rename columns
external_all_df.columns = [
    "GT",
    "Zscore",
    "Rescale",
    "Clip Rescale",
    "Hist Eq",
    "Clahe",
    "Nyul",
    "GMM"
]

In [ ]:
external_all_df.describe()

In [ ]:
# plot pairwise correlation matrix of dice scores, to 3 decimal places
plt.figure(figsize=(10, 6))
sns.heatmap(external_all_df.corr('pearson'), annot=True, cmap='coolwarm', fmt=".3f")
plt.title("Pearson correlation matrix of Dice scores")
plt.show()

In [ ]:
# bland-altman plot for zscore and zscore hist match
sm.graphics.mean_diff_plot(external_all_df["GT"], external_all_df["Zscore"])
plt.show()

In [ ]:
# calculate CCC and MAE for each method
ccc_scores = []
mae_scores = []

for method in column_names[1:]:
    ccc_scores.append(ccc(external_all_df["GT"], external_all_df[method]))
    mae_scores.append(mean_absolute_error(external_all_df["GT"], external_all_df[method]))

# create dataframe of scores
external_scores_df = pd.DataFrame({
    "Method": column_names[1:],
    "CCC": ccc_scores,
    "MAE": mae_scores
})

external_scores_df

### Thickness

In [ ]:
# get skmtea spacing

skmtea_mask = sitk.ReadImage(os.path.join('../nnUNet_data/nnUNet_raw/Dataset361_Menisci/labels_all_skmtea', 'SKMTEA_001.nii.gz'))
skmtea_spacing = np.array(skmtea_mask.GetSpacing())[::-1]
print(skmtea_spacing)

In [ ]:
# calculate average thickness for all GT masks
gt_masks_paths = glob.glob(os.path.join(data_path, '../labels_all_skmtea', '*.nii.gz'))
gt_masks_paths.sort()

average_thicknesses = []

for mask_path in gt_masks_paths:
    mask = sitk.ReadImage(mask_path)
    mask_np = sitk.GetArrayFromImage(mask)
    np_spacing = np.array(mask.GetSpacing())[::-1]
    thickness = average_thickness(mask_np, np_spacing)
    average_thicknesses.append(thickness)

# create dataframe of average thicknesses
thickness_df = pd.DataFrame({
    "GT": average_thicknesses
})

# now calculate average thickness for all predicted masks of all methods
for folder in tqdm(subfolders):
    masks_paths = glob.glob(os.path.join(data_path, folder) + '/*.nii.gz')
    masks_paths.sort()

    average_thicknesses = []

    for mask_path in masks_paths:
        mask = sitk.ReadImage(mask_path)
        mask_np = sitk.GetArrayFromImage(mask)
        np_spacing = np.array(mask.GetSpacing())[::-1]
        thickness = average_thickness(mask_np, np_spacing)
        average_thicknesses.append(thickness)

    thickness_df[folder] = average_thicknesses

In [ ]:
# rename columns
thickness_df.columns = column_names

# describe dataframe
thickness_df.describe()

In [ ]:
# do CCC and MAE for average thickness
ccc_scores = []
mae_scores = []

for method in column_names[1:]:
    ccc_scores.append(ccc(thickness_df["GT"], thickness_df[method]))
    mae_scores.append(mean_absolute_error(thickness_df["GT"], thickness_df[method]))

# create dataframe of scores
thickness_scores_df = pd.DataFrame({
    "Method": column_names[1:],
    "CCC": ccc_scores,
    "MAE": mae_scores
})

thickness_scores_df

In [ ]:
# bland-altman plot for GT and Nyul
sm.graphics.mean_diff_plot(thickness_df["GT"], thickness_df["GMM"])
plt.show()

In [ ]:
# mae as a percentage of the mean thickness
for method in column_names[1:]:
    mean_thickness = np.mean(thickness_df["GT"])
    mae = mean_absolute_error(thickness_df["GT"], thickness_df[method])
    mae_percentage = (mae / mean_thickness) * 100
    print(f"{method}: {mae_percentage:.2f}%")

In [ ]:
# mae as a percentage of the mean volume
for method in column_names[1:]:
    mean_volume = np.mean(external_all_df["GT"])
    mae = mean_absolute_error(external_all_df["GT"], external_all_df[method])
    mae_percentage = (mae / mean_volume) * 100
    print(f"{method}: {mae_percentage:.2f}%")

## Fold results internal

In [ ]:
# now we want to calculate ccc and mae for each fold of each method to get a sense of variability
# get all subfolders in the data path
print(subfolders)

# create empty dictionary to hold internal results
ccc_dict = {}
mae_dict = {}

for i, folder in enumerate(subfolders):

    fold_ccc = []
    fold_mae = []

    # now cycle through fold folders
    for fold in range(5):
        # create json path
        json_path = os.path.join(data_path, folder, f'fold{fold}', 'summary.json')

        # get ground truth volumes only for the first iteration
        if i == 0 and fold == 0:
            gt_vols = np.array(get_gt_vols(json_path))
        
        # now get segmentation volumes for each fold of each method
        seg_vols = np.array(get_seg_vols(json_path))

        # calculate ccc and mae for this fold
        fold_ccc.append(ccc(gt_vols, seg_vols))
        fold_mae.append(mean_absolute_error(gt_vols, seg_vols))

    # add to dictionary
    ccc_dict[folder] = fold_ccc
    mae_dict[folder] = fold_mae


In [ ]:
ccc_df = pd.DataFrame(ccc_dict)
ccc_df.describe()

In [ ]:
sns.pointplot(data=ccc_df, markers='o', linestyles='none', errorbar='sd')
plt.xticks(rotation=45)
plt.show()

In [ ]:
mae_df = pd.DataFrame(mae_dict)

# convert to mm
mae_df = mae_df * np.prod(iwoai_spacing)
mae_df.describe()

In [ ]:
sns.set(style="whitegrid")
sns.pointplot(data=mae_df, markers='o', linestyles='none', errorbar='sd')
plt.xticks(rotation=45)
plt.ylabel('Mean Absolute Error (mm)')
plt.show()

## Fold results external

In [ ]:
# do same for external results
# create empty dictionary to hold external results
ccc_dict_external = {}
mae_dict_external = {}

for i, folder in enumerate(subfolders):

    fold_ccc = []
    fold_mae = []

    # now cycle through fold folders
    for fold in range(5):
        # create json path
        json_path = os.path.join(data_path, folder, f'fold{fold}', 'summary.json')

        # get ground truth volumes only for the first iteration
        if i == 0 and fold == 0:
            gt_vols = np.array(get_gt_vols(json_path))

        # now get segmentation volumes for each fold of each method
        seg_vols = np.array(get_seg_vols(json_path))

        # calculate ccc and mae for this fold
        fold_ccc.append(ccc(gt_vols, seg_vols))
        fold_mae.append(mean_absolute_error(gt_vols, seg_vols))

    # add to dictionary
    ccc_dict_external[folder] = fold_ccc
    mae_dict_external[folder] = fold_mae
ccc_df_external = pd.DataFrame(ccc_dict_external)
mae_df_external = pd.DataFrame(mae_dict_external)

In [ ]:
ccc_df_external.describe()

In [ ]:
sns.set(style="whitegrid")
sns.pointplot(data=ccc_df_external, markers='o', linestyles='none', errorbar='sd', capsize=0.2)
plt.xticks(rotation=45)
plt.ylabel('CCC')
plt.show()

In [ ]:
mae_df_external = mae_df_external * np.prod(skmtea_spacing)
mae_df_external.describe()

In [ ]:
sns.set(style="whitegrid")
sns.pointplot(data=mae_df_external, markers='o', linestyles='none', errorbar='sd', capsize=0.2)
plt.xticks(rotation=45)
plt.ylabel('Mean Absolute Error (mm)')
plt.show()